# Notebook 02: Data Cleaning, Harmonization & Exploratory Data Analysis
## EduEquity LK: Uncovering Regional and Sectoral Education Disparities

### Notebook Objectives
1. Harmonize district nomenclature and merge multi-year school census data with socioeconomic indicators.
2. Analyze longitudinal time-series trends (2014–2024) and isolate the impact of the 2022 macroeconomic crisis.
3. Investigate the persistent disparity between the **Estate (Tea Plantation)** sector and the Urban/Rural sectors.
4. Examine the Grade-by-Grade survival curve to identify critical educational drop-off transitions.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

project_root = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
sys.path.insert(0, os.path.abspath(project_root))

from src.data_pipeline import DataPipeline
from src.utils import PALETTE, format_pct

pipeline = DataPipeline(
    raw_dir=os.path.join(project_root, "data/raw"),
    processed_dir=os.path.join(project_root, "data/processed")
)
data_dict = pipeline.process_and_save_all()
master_df = data_dict["master_timeseries"]
grade_df = data_dict["grade_matrix"]
sector_df = data_dict["sector_benchmark"]

print("Master dataset shape:", master_df.shape)


### 1. Longitudinal Dropout Trends (2014–2024)
Let us examine the evolution of annual dropout rates across administrative districts and observe the sharp surge in 2022 driven by paper shortages, transportation costs, and soaring household poverty.


In [ ]:
# National time series aggregate
ts_national = master_df.groupby("year").agg({
    "total_enrolment": "sum",
    "dropout_count": "sum"
}).reset_index()
ts_national["national_dropout_rate_pct"] = (ts_national["dropout_count"] / ts_national["total_enrolment"]) * 100

print("National Time-Series Summary:")
print(ts_national[["year", "total_enrolment", "dropout_count", "national_dropout_rate_pct"]])


### 2. Estate vs Urban vs Rural Sector Disparities
Data from the Department of Census and Statistics (DCS) Child Activity Survey reveals stark structural divides.


In [ ]:
print("Sector Equity Benchmark:")
print(sector_df[["sector", "currently_attending_school_pct", "not_attending_never_attended_pct", "child_labour_rate_pct", "reason_poverty_pct", "avg_monthly_hh_income_lkr"]])


### 3. Grade Progression and the Drop-Off Cliff
We inspect the dropout rate at each curriculum grade level from Grade 1 through Grade 11 (G.C.E. O/L).


In [ ]:
print(grade_df[["grade_level", "stage", "urban_dropout_rate_pct", "rural_dropout_rate_pct", "estate_dropout_rate_pct", "estate_to_urban_disparity_ratio"]])
